# Text Classification Basics

This notebook builds a small sentiment classifier from scratch. It first demonstrates the text-processing pipeline with a toy dataset, then follows a proper train, validation, and test workflow.

In [1]:
import re
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## 1. Prepare toy data

The initial dataset contains short positive and negative sentences. This section is a compact demonstration of the full text-to-model pipeline.

In [2]:
sentences = [
    "i love this movie",
    "this product is amazing",
    "the food was great",
    "i really like this book",
    "i hate this movie",
    "this product is terrible",
    "the food was bad",
    "i really dislike this book",
]

labels = torch.tensor([
    1,  # positive
    1,
    1,
    1,
    0,  # negative
    0,
    0,
    0,
])

print(sentences)
print(labels)

['i love this movie', 'this product is amazing', 'the food was great', 'i really like this book', 'i hate this movie', 'this product is terrible', 'the food was bad', 'i really dislike this book']
tensor([1, 1, 1, 1, 0, 0, 0, 0])


## 2. Tokenize text and build a vocabulary

Text is converted into lowercase word tokens. A vocabulary then assigns each token an integer ID, while special tokens represent padding and unknown words.

In [3]:
def tokenize(text):
    normalized_text = text.lower()
    return re.findall(r"[a-z]+", normalized_text)

tokenized_sentences = [tokenize(sentence) for sentence in sentences]

print(tokenized_sentences)

[['i', 'love', 'this', 'movie'], ['this', 'product', 'is', 'amazing'], ['the', 'food', 'was', 'great'], ['i', 'really', 'like', 'this', 'book'], ['i', 'hate', 'this', 'movie'], ['this', 'product', 'is', 'terrible'], ['the', 'food', 'was', 'bad'], ['i', 'really', 'dislike', 'this', 'book']]


### Encode token IDs

Each token list is encoded as integer IDs. Tokens missing from the vocabulary are mapped to the unknown-token ID.

In [4]:
vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

for tokens in tokenized_sentences:
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

print(vocab)
print("Vocabulary size:", len(vocab))

{'<PAD>': 0, '<UNK>': 1, 'i': 2, 'love': 3, 'this': 4, 'movie': 5, 'product': 6, 'is': 7, 'amazing': 8, 'the': 9, 'food': 10, 'was': 11, 'great': 12, 'really': 13, 'like': 14, 'book': 15, 'hate': 16, 'terrible': 17, 'bad': 18, 'dislike': 19}
Vocabulary size: 20


### Pad sequences

Padding makes every sentence in a batch the same length, allowing them to be stored in a single tensor.

In [5]:
def encode(tokens):
    return [
        vocab.get(token, vocab["<UNK>"])
        for token in tokens
    ]

encoded_sentences = [
    encode(tokens)
    for tokens in tokenized_sentences
]

print(encoded_sentences)

[[2, 3, 4, 5], [4, 6, 7, 8], [9, 10, 11, 12], [2, 13, 14, 4, 15], [2, 16, 4, 5], [4, 6, 7, 17], [9, 10, 11, 18], [2, 13, 19, 4, 15]]


## 3. Create a DataLoader and inspect embeddings

The DataLoader groups token-ID sequences and labels into batches. An embedding layer then converts each token ID into a learned vector.

In [6]:
max_length = max(len(sentence) for sentence in encoded_sentences)

pad_id = vocab["<PAD>"]
padded_sentences = []

for ids in encoded_sentences:
    padding_needed = max_length - len(ids)

    padded_ids = ids + [pad_id] * padding_needed

    padded_sentences.append(padded_ids)

input_ids = torch.tensor(padded_sentences)

print(input_ids)
print(input_ids.shape)

tensor([[ 2,  3,  4,  5,  0],
        [ 4,  6,  7,  8,  0],
        [ 9, 10, 11, 12,  0],
        [ 2, 13, 14,  4, 15],
        [ 2, 16,  4,  5,  0],
        [ 4,  6,  7, 17,  0],
        [ 9, 10, 11, 18,  0],
        [ 2, 13, 19,  4, 15]])
torch.Size([8, 5])


### Embedding lookup

The embedding table preserves the batch and sequence dimensions, then adds an embedding dimension for every token.

In [7]:
dataset = TensorDataset(input_ids, labels)

dataloader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True,
)

x_batch, y_batch = next(iter(dataloader))

print(x_batch.shape)
print(y_batch.shape)

torch.Size([2, 5])
torch.Size([2])


### Padding-aware mean pooling

A mask excludes padding tokens before averaging token embeddings into one vector for each sentence.

In [8]:
embedding = nn.Embedding(
    num_embeddings=len(vocab),
    embedding_dim=8,
    padding_idx=pad_id,
)

embedded = embedding(x_batch)

print(embedded.shape)

torch.Size([2, 5, 8])


## 4. Build the sentiment classifier

The classifier combines an embedding layer, padding-aware mean pooling, and a linear layer that produces one logit for binary sentiment classification.

In [9]:
mask = (x_batch != pad_id).unsqueeze(-1)
masked_embeddings = embedded * mask
embedding_sum = masked_embeddings.sum(dim=1)

word_count = mask.sum(dim=1)

sentence_vectors = embedding_sum / word_count

print(sentence_vectors.shape)

torch.Size([2, 8])


## 5. Train and evaluate the prototype

BCEWithLogitsLoss is used for binary labels. The model output is a logit, which is converted to a probability with sigmoid during evaluation.

In [10]:
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, pad_id):
        super().__init__()

        self.pad_id = pad_id
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.classifier = nn.Linear(embedding_dim, 1)

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)

        mask = (input_ids != self.pad_id).unsqueeze(-1)
        masked_embeddings = embedded * mask
        embedding_sum = masked_embeddings.sum(dim=1)

        word_count = mask.sum(dim=1)

        sentence_vectors = embedding_sum / word_count

        logits = self.classifier(sentence_vectors)

        return logits.squeeze(1)
                

### Training helper

This function completes one pass over the DataLoader and returns the average loss across its batches.

In [11]:
model = SentimentClassifier(
    vocab_size=len(vocab),
    embedding_dim=8,
    pad_id=pad_id
)

logits = model(x_batch)

print(logits)
print(logits.shape)

tensor([0.0490, 0.0647], grad_fn=<SqueezeBackward1>)
torch.Size([2])


### Train the prototype

The prototype is trained on the complete toy dataset. Its accuracy only confirms that the pipeline works; it is not a valid measure of generalization.

In [12]:
loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01,
)

### Accuracy helper

Logits are converted to probabilities with sigmoid, then thresholded at 0.5 to obtain positive or negative predictions.

In [13]:
def train_one_epoch(dataloader, model, loss_fn, optimizer):
    model.train()
    total_loss = batch_count = 0

    for x_batch, y_batch in dataloader:
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        batch_count += 1

    return total_loss / batch_count

### Prototype training accuracy

Because this score is calculated on data used for training, it should not be treated as a final performance metric.

In [14]:
epochs = 100

for epoch in range(epochs):
    loss = train_one_epoch(dataloader, model, loss_fn, optimizer)

    if ((epoch + 1) % 20 == 0):
        print(f"Epoch {epoch + 1}: loss = {loss:.4f}")

Epoch 20: loss = 0.6010
Epoch 40: loss = 0.3485
Epoch 60: loss = 0.1227
Epoch 80: loss = 0.0498
Epoch 100: loss = 0.0260


## 6. Run inference on new text

New text must follow the same tokenization, encoding, truncation, and padding steps before it can be passed to the model.

In [15]:
def test_accuracy(dataloader, model):
    model.eval()
    correct = total = 0

    with torch.no_grad():
        for x_batch, y_batch in dataloader:
            logits = model(x_batch)
            probabilities = torch.sigmoid(logits)
            predicted = (probabilities >= 0.5).long()
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    return correct / total

In [16]:
print(test_accuracy(dataloader, model))

1.0


In [18]:
def prepare_text(text):
    tokens = tokenize(text)
    token_ids = encode(tokens)

    token_ids = token_ids[:max_length]

    padding_needed = max_length - len(token_ids)
    padded_ids = token_ids + [pad_id] * padding_needed

    return torch.tensor([padded_ids])

In [19]:
new_input = prepare_text("i love this product")

print(new_input)
print(new_input.shape)

tensor([[2, 3, 4, 6, 0]])
torch.Size([1, 5])


In [20]:
def predict_sentiment(text):
    input_ids = prepare_text(text)

    model.eval()

    with torch.no_grad():
        logits = model(input_ids)
        probabilities = torch.sigmoid(logits)
        predicted = (probabilities >= 0.5).long()

        if (predicted.item()):
            return "positive"
        else:
            return "negative"
        

In [21]:
print(predict_sentiment("i love this product"))
print(predict_sentiment("this movie was terrible"))

positive
negative


## 7. Train, validation, and test workflow

The following sections rebuild the pipeline with separate training, validation, and test data. The vocabulary is created from training text only to prevent data leakage.

In [22]:
train_sentences = [
    "i love this movie",
    "this product is amazing",
    "the food was great",
    "i hate this movie",
    "this product is terrible",
    "the food was bad",
]

train_labels = torch.tensor([1, 1, 1, 0, 0, 0])

validation_sentences = [
    "i love this product",
    "this movie was bad",
]

validation_labels = torch.tensor([1, 0])

### Build a vocabulary from training data only

Validation and test sentences must not contribute tokens to the vocabulary, because they represent unseen evaluation data.

In [23]:
train_tokenized = [tokenize(sentence) for sentence in train_sentences]

print(train_tokenized)

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

for tokens in train_tokenized:
    for token in tokens:
        if token not in vocab:
            vocab[token] = len(vocab)

print(vocab)
print("Vocabulary size:", len(vocab))

[['i', 'love', 'this', 'movie'], ['this', 'product', 'is', 'amazing'], ['the', 'food', 'was', 'great'], ['i', 'hate', 'this', 'movie'], ['this', 'product', 'is', 'terrible'], ['the', 'food', 'was', 'bad']]
{'<PAD>': 0, '<UNK>': 1, 'i': 2, 'love': 3, 'this': 4, 'movie': 5, 'product': 6, 'is': 7, 'amazing': 8, 'the': 9, 'food': 10, 'was': 11, 'great': 12, 'hate': 13, 'terrible': 14, 'bad': 15}
Vocabulary size: 16


### Encode and pad each split

All splits use the training vocabulary and training maximum sequence length. Longer sequences are truncated and shorter ones are padded.

In [25]:
train_encoded = [
    encode(tokens)
    for tokens in train_tokenized
]

validation_tokenized = [
    tokenize(sentence)
    for sentence in validation_sentences
]

validation_encoded = [
    encode(tokens)
    for tokens in validation_tokenized
]

max_length = max(len(sentence) for sentence in train_encoded)
pad_id = vocab["<PAD>"]

def pad_sequences(encoded_sentences):
    padded_sentences = []

    for ids in encoded_sentences:
        ids = ids[:max_length]

        padding_needed = max_length - len(ids)
        padded_ids = ids + [pad_id] * padding_needed

        padded_sentences.append(padded_ids)

    return torch.tensor(padded_sentences)

train_input_ids = pad_sequences(train_encoded)
validation_input_ids = pad_sequences(validation_encoded)

print(train_input_ids.shape)
print(validation_input_ids.shape)

torch.Size([6, 4])
torch.Size([2, 4])


### Create DataLoaders

The training DataLoader shuffles examples, while validation and test DataLoaders preserve a fixed order.

In [26]:
train_dataset = TensorDataset(
    train_input_ids,
    train_labels
)

validation_dataset = TensorDataset(
    validation_input_ids,
    validation_labels
)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True
)

validation_dataloader = DataLoader(
    validation_dataset,
    batch_size=2,
    shuffle=False
)

### Create a fresh model

A new model is required because the vocabulary and training split have changed from the earlier prototype.

In [27]:
model = SentimentClassifier(
    vocab_size=len(vocab),
    embedding_dim=8,
    pad_id=pad_id
)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

### Train with validation

Each epoch trains on the training DataLoader and measures accuracy on the validation DataLoader. The validation set guides development without using the final test set.

In [28]:
epochs = 100

for epoch in range(epochs):
    loss = train_one_epoch(train_dataloader, model, loss_fn, optimizer)
    accuracy = test_accuracy(validation_dataloader, model)
    if ((epoch + 1) % 20 == 0):
        print(f"Epoch {epoch + 1}: loss = {loss:.4f}, validation accuracy = {accuracy:.2%}")

Epoch 20: loss = 0.6013, validation accuracy = 50.00%
Epoch 40: loss = 0.3446, validation accuracy = 100.00%
Epoch 60: loss = 0.1314, validation accuracy = 100.00%
Epoch 80: loss = 0.0551, validation accuracy = 100.00%
Epoch 100: loss = 0.0293, validation accuracy = 100.00%


## 8. Final test evaluation

The test set remains separate until the end. It is encoded with the training vocabulary and evaluated only after model development is complete.

In [29]:
test_sentences = [
    "the product was amazing",
    "i hate this food",
]

test_labels = torch.tensor([1, 0])

In [30]:
test_tokenized = [tokenize(sentence) for sentence in test_sentences]

test_encoded = [
    encode(tokens)
    for tokens in test_tokenized
]

test_input_ids = pad_sequences(test_encoded)

test_dataset = TensorDataset(
    test_input_ids,
    test_labels
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False
)

In [31]:
final_test_accuracy = test_accuracy(test_dataloader, model)

print(f"Final test accuracy: {final_test_accuracy:.2%}")

Final test accuracy: 100.00%


## Result

Final test accuracy: **100.00%**. This result is based on a two-example toy test set, so it validates the workflow rather than demonstrating real-world model performance.